In [0]:
# BRONZE - Ingesta de datos crudos
from pyspark.sql import *
from pyspark.sql.functions import *

In [0]:
# Acceder al secreto almacenado
storage_path = dbutils.secrets.get(scope="tpfinal", key="keyfinal")

dbutils.fs.ls(storage_path)

[FileInfo(path='[REDACTED]/bronze/', name='bronze/', size=0, modificationTime=1761776216000),
 FileInfo(path='[REDACTED]/raw/', name='raw/', size=0, modificationTime=1761678416000),
 FileInfo(path='[REDACTED]/silver/', name='silver/', size=0, modificationTime=1761833611000),
 FileInfo(path='[REDACTED]/vuelos/', name='vuelos/', size=0, modificationTime=1761611410000)]

In [0]:
# volcar datos de vuelos a dataframe

from pyspark.sql import functions as F

base_vuelos = f"{storage_path}/vuelos"

df_vuelos_raw = (spark.read
    .option("recursiveFileLookup", "true")
    .option("pathGlobFilter", "all.json")  
    .option("multiLine", True)
    .json(base_vuelos)
    .filter(~F.lower(F.input_file_name()).rlike(r"all-keys\.json$"))
    .withColumn("ingest_date",
        F.to_date(F.regexp_extract(F.input_file_name(),
                                   r"(\d{4}-\d{2}-\d{2})", 1))
    )
)

df_vuelos_raw.printSchema()
display(df_vuelos_raw.limit(10))


root
 |-- IATAdestorig: string (nullable = true)
 |-- acft_body: string (nullable = true)
 |-- acftype: string (nullable = true)
 |-- aerolinea: string (nullable = true)
 |-- arpt: string (nullable = true)
 |-- atda: string (nullable = true)
 |-- belt: string (nullable = true)
 |-- blockoff: string (nullable = true)
 |-- blockon: string (nullable = true)
 |-- checkins: string (nullable = true)
 |-- chk_from: string (nullable = true)
 |-- chk_lyf: string (nullable = true)
 |-- chk_to: string (nullable = true)
 |-- color: string (nullable = true)
 |-- destorig: string (nullable = true)
 |-- estbr: string (nullable = true)
 |-- estes: string (nullable = true)
 |-- estin: string (nullable = true)
 |-- etda: string (nullable = true)
 |-- gate: string (nullable = true)
 |-- id: string (nullable = true)
 |-- id_flight_reg: string (nullable = true)
 |-- id_flight_tp: string (nullable = true)
 |-- id_flight_tra: string (nullable = true)
 |-- idaerolinea: string (nullable = true)
 |-- idclimaico

IATAdestorig,acft_body,acftype,aerolinea,arpt,atda,belt,blockoff,blockon,checkins,chk_from,chk_lyf,chk_to,color,destorig,estbr,estes,estin,etda,gate,id,id_flight_reg,id_flight_tp,id_flight_tra,idaerolinea,idclimaicono,idshared,logo,matricula,mov,nro,pasajeros,posicion,rot,sdphrase,sdtemp,sdtempunit,sector,stda,term,termsec,tipoVuelo,via,ingest_date
PMY,NB,null,AEROLINEAS ARGENTINAS,AEP,30/12 01:01,4,30/12 03:45,30/12 01:05,,null,null,null,#008000,Puerto Madryn,Pousado,Aterrizado,Landed,30/12 00:55,,7662570,C,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,LVGUC,A,AR 1815,133,06,AR 1872,null,null,null,N,29/12 23:00,null,AN,null,,2024-12-31
MDZ,NB,null,JETSMART AIRLINES,AEP,30/12 00:02,7,30/12 05:16,30/12 00:04,,null,null,null,#008000,Mendoza,Pousado,Aterrizado,Landed,29/12 23:59,,7667738,C,1,P,WJ,null,,https://www.aeropuertosargentina.com/img/aerolineas/WJ_200.GIF,LVIVO,A,WJ 3075,142,09,WJ 3100,null,null,null,N,30/12 00:04,null,AN,null,,2024-12-31
COR,NB,null,JETSMART AIRLINES,AEP,30/12 00:11,8,30/12 05:43,30/12 00:21,,null,null,null,#008000,Córdoba,Pousado,Aterrizado,Landed,30/12 00:14,,7666327,C,1,P,WJ,null,,https://www.aeropuertosargentina.com/img/aerolineas/WJ_200.GIF,CCAWA,A,WJ 3111,,10,WJ 3012,null,null,null,N,30/12 00:33,null,AN,null,,2024-12-31
SCL,NB,null,AEROLINEAS ARGENTINAS,AEP,30/12 01:05,2I,30/12 05:28,30/12 01:13,,null,null,null,#008000,Santiago de Chile,Pousado,Aterrizado,Landed,30/12 01:05,,7666450,I,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,LVGVA,A,AR 1289,,05,AR 1280,null,null,null,I,30/12 00:50,null,AI,null,,2024-12-31
SCL,NB,null,JETSMART AIRLINES,AEP,30/12 00:57,1I,30/12 04:49,30/12 01:01,,null,null,null,#008000,Santiago de Chile,Pousado,Aterrizado,Landed,30/12 01:05,,7664272,I,1,P,WJ,null,,https://www.aeropuertosargentina.com/img/aerolineas/WJ_200.GIF,LVHEK,A,WJ 3885,,08,WJ 3161,null,null,null,I,30/12 01:20,null,AI,null,,2024-12-31
BSB,NB,null,AEROLINEAS ARGENTINAS,AEP,30/12 01:13,3I,30/12 05:36,30/12 01:25,,null,null,null,#008000,Brasilia,Pousado,Aterrizado,Landed,30/12 01:15,,7666451,I,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,LVFYK,A,AR 1219,,31,AR 1694,null,null,null,I,30/12 01:30,null,AI,null,,2024-12-31
GRU,NB,null,GOL,AEP,30/12 01:26,1I,30/12 02:46,30/12 01:57,,null,null,null,#008000,San Pablo-Guarulhos,Pousado,Aterrizado,Landed,30/12 01:33,,7664546,I,1,P,G3,null,,https://www.aeropuertosargentina.com/img/aerolineas/G3_200.GIF,PRXMP,A,G3 7650,,03,G3 7651,null,null,null,I,30/12 02:00,null,AI,null,,2024-12-31
GRU,NB,null,AEROLINEAS ARGENTINAS,AEP,30/12 02:35,1I,30/12 06:10,30/12 02:47,,null,null,null,#008000,San Pablo-Guarulhos,Pousado,Aterrizado,Landed,30/12 02:44,,7664529,I,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,LVHKW,A,AR 1319,,28,AR 1238,null,null,null,I,30/12 02:45,null,AI,null,,2024-12-31
SCL,NB,null,SKY AIRLINE,AEP,30/12 03:33,2I,30/12 04:48,30/12 03:36,,null,null,null,#008000,Santiago de Chile,Pousado,Aterrizado,Landed,30/12 03:24,,7665905,I,1,P,H2,null,,https://www.aeropuertosargentina.com/img/aerolineas/H2_200.GIF,CCAZE,A,H2 5572,100,03,H2 5572,null,null,null,I,30/12 03:40,null,AI,null,,2024-12-31
GRU,NB,null,AEROLINEAS ARGENTINAS,AEP,30/12 03:29,3I,30/12 06:52,30/12 03:33,,null,null,null,#008000,San Pablo-Guarulhos,Pousado,Aterrizado,Landed,30/12 03:34,,7664563,I,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,LVKKD,A,AR 1251,,04,AR 1226,null,null,null,I,30/12 04:05,null,AI,null,,2024-12-31


In [0]:
# crear tabla vuelos dentro de la capa bronze
bronze_delta_vuelos = f"{storage_path}/bronze/delta/vuelos"

(df_vuelos_raw.write
   .format("delta")
   .mode("overwrite")
   .option("overwriteSchema","true")
   .save(bronze_delta_vuelos))

# Crear el schema (si no existe)
spark.sql(f"""
          CREATE SCHEMA IF NOT EXISTS capa_bronze
          """)

# Crear la tabla delta apuntando al path
spark.sql(f"""
CREATE TABLE IF NOT EXISTS capa_bronze.vuelos_delta
USING delta
LOCATION '{bronze_delta_vuelos}'
""")


DataFrame[]

In [0]:
%sql
select * from capa_bronze.vuelos_delta limit 10

IATAdestorig,acft_body,acftype,aerolinea,arpt,atda,belt,blockoff,blockon,checkins,chk_from,chk_lyf,chk_to,color,destorig,estbr,estes,estin,etda,gate,id,id_flight_reg,id_flight_tp,id_flight_tra,idaerolinea,idclimaicono,idshared,logo,matricula,mov,nro,pasajeros,posicion,rot,sdphrase,sdtemp,sdtempunit,sector,stda,term,termsec,tipoVuelo,via,ingest_date
AEP,NB,null,AEROLINEAS ARGENTINAS,CTC,,,,,004-006,null,null,null,#C0C0C0,Aeroparque,No Horario,En Horario,On Time,,1,7652552,C,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,,D,AR 1455,,001,AR 1454,null,null,null,P,23/12 11:00,null,P,null,,2024-12-23
AEP,NB,null,AEROLINEAS ARGENTINAS,CTC,,,,,004-006,null,null,null,#C0C0C0,Aeroparque,No Horario,En Horario,On Time,,1,7652553,C,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,,D,AR 1458,,001,AR 1458,null,null,null,P,23/12 18:55,null,P,null,IRJ,2024-12-23
AEP,NB,null,AEROLINEAS ARGENTINAS,CRD,,,,,001-004,null,null,null,#C0C0C0,Aeroparque,No Horario,En Horario,On Time,,03,7652548,C,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,,D,AR 1837,,02,AR 1836,null,null,null,P,23/12 01:50,null,P,null,,2024-12-23
AEP,NB,null,AEROLINEAS ARGENTINAS,CRD,,,,,001-004,null,null,null,#C0C0C0,Aeroparque,No Horario,En Horario,On Time,,03,7652545,C,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,,D,AR 1823,,01,AR 1822,null,null,null,P,23/12 08:10,null,P,null,,2024-12-23
USH,NB,null,LADE,CRD,,,,,005-006,null,null,null,#C0C0C0,Ushuaia,No Horario,En Horario,On Time,,01,7652536,C,1,P,5U,null,,https://www.aeropuertosargentina.com/img/aerolineas/5U_200.GIF,,D,5U 444,,03A,,null,null,null,P,23/12 08:30,null,P,null,"PMQ,FTE,RGL",2024-12-23
NQN,NB,null,AEROLINEAS ARGENTINAS,CRD,,,,,001-004,null,null,null,#C0C0C0,Neuquén,No Horario,En Horario,On Time,,03,7652543,C,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,,D,AR 1431,,02,AR 1824,null,null,null,P,23/12 11:15,null,P,null,,2024-12-23
COR,NB,null,AEROLINEAS ARGENTINAS,CRD,,,,,001-004,null,null,null,#C0C0C0,Córdoba,No Horario,En Horario,On Time,,03,7652544,C,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,,D,AR 1559,,01,AR 1828,null,null,null,P,23/12 13:00,null,P,null,,2024-12-23
AEP,NB,null,FLYBONDI,CRD,,,,,007-008,null,null,null,#C0C0C0,Aeroparque,No Horario,En Horario,On Time,,04,7652550,C,1,P,FO,null,,https://www.aeropuertosargentina.com/img/aerolineas/FO_200.GIF,,D,FO 5501,,01,FO 5500,null,null,null,P,23/12 14:45,null,P,null,,2024-12-23
AEP,NB,null,AEROLINEAS ARGENTINAS,CRD,,,,,001-004,null,null,null,#C0C0C0,Aeroparque,No Horario,En Horario,On Time,,03,7652546,C,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,,D,AR 1829,,02,AR 1558,null,null,null,P,23/12 16:20,null,P,null,,2024-12-23
AEP,NB,null,AEROLINEAS ARGENTINAS,CRD,,,,,001-004,null,null,null,#C0C0C0,Aeroparque,No Horario,En Horario,On Time,,03,7652547,C,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,,D,AR 1835,,01,AR 1834,null,null,null,P,23/12 20:55,null,P,null,,2024-12-23


In [0]:
# lectura de feriados
path_feriados_raw = f"{storage_path}/raw/feriados"
df_feriados_raw = (spark.read
    .option("recursiveFileLookup", "true")
    .option("multiLine", False)
    .json(path_feriados_raw)
)

bronze_delta_feriados = f"{storage_path}/bronze/delta/feriados"

(df_feriados_raw.write
   .format("delta")
   .mode("overwrite")
   .option("overwriteSchema","true")
   .save(bronze_delta_feriados))

spark.sql(f"""
CREATE TABLE IF NOT EXISTS capa_bronze.feriados_delta
USING delta
LOCATION '{bronze_delta_feriados}'
""")

DataFrame[]

In [0]:
%sql
SELECT * from capa_bronze.feriados_delta

fecha,nombre,tipo
2024-01-01,Año nuevo,inamovible
2024-02-12,Carnaval,inamovible
2024-02-13,Carnaval,inamovible
2024-03-24,Día Nacional de la Memoria por la Verdad y la Justicia,inamovible
2024-03-29,Viernes Santo,inamovible
2024-04-01,Feriado puente turístico,puente
2024-04-02,Día del Veterano y de los Caídos en la Guerra de Malvinas,inamovible
2024-05-01,Día del Trabajador,inamovible
2024-05-25,Día de la Revolución de Mayo,inamovible
2024-06-17,Paso a la Inmortalidad del General Martín Güemes,trasladable


In [0]:
# Para borrar los datos

# storage_path = dbutils.secrets.get(scope="tpfinal", key="keyfinal")

# for capa in ["bronze", "silver", "gold"]:
#     path = f"{storage_path}/{capa}"
#     print(f"Borrando {path} ...")
#     try:
#         dbutils.fs.rm(path, recurse=True)
#         print(f"✅ {capa} borrado")
#     except Exception as e:
#         print(f"⚠️ {capa} no encontrado o ya vacío: {e}")


Borrando [REDACTED]/bronze ...
⚠️ bronze no encontrado o ya vacío: [RequestId=f231cf12-4528-43e7-91ec-6aa2f9901b81 ErrorClass=INVALID_PARAMETER_VALUE.LOCATION_OVERLAP] Input path url '[REDACTED]/bronze' overlaps with other external tables or volumes within 'CheckPathAccess' call. Conflicting tables/volumes: tpfinal2025.capa_bronze.feriados_delta, tpfinal2025.capa_bronze.vuelos_delta.
Borrando [REDACTED]/silver ...
⚠️ silver no encontrado o ya vacío: [RequestId=03843130-e08e-48d5-b4ba-d8ba74ae076a ErrorClass=INVALID_PARAMETER_VALUE.LOCATION_OVERLAP] Input path url '[REDACTED]/silver' overlaps with other external tables or volumes within 'CheckPathAccess' call. Conflicting tables/volumes: tpfinal2025.capa_silver.feriados, tpfinal2025.capa_silver.vuelos.
Borrando [REDACTED]/gold ...
⚠️ gold no encontrado o ya vacío: [RequestId=595297d8-c6e9-4c57-8513-aa9fab2a28d3 ErrorClass=INVALID_PARAMETER_VALUE.LOCATION_OVERLAP] Input path url '[REDACTED]/gold' overlaps with other external tables or vo

In [0]:
# para borrar las tablas de los catalogos

# for db in ["capa_bronze", "capa_silver", "capa_gold"]:
#     try:
#         spark.sql(f"DROP DATABASE IF EXISTS {db} CASCADE")
#         print(f"✅ {db} eliminado")
#     except Exception as e:
#         print(f"⚠️ Error en {db}: {e}")


✅ capa_bronze eliminado
✅ capa_silver eliminado
✅ capa_gold eliminado
